# 4 — DP-SGD Fine-Tuning (Opacus)

Train GPT-2 small on each canary-augmented dataset across all (ε, frequency) conditions.
For ε = ∞ we train without DP; for all other ε values we use Opacus `PrivacyEngine`.

Grid: **6 ε values × 4 frequencies = 24 training runs.**

| ε | Description |
|---|-------------|
| 0.5, 1, 2, 4, 8 | DP-SGD (Opacus) |
| ∞ | No DP (baseline with canaries) |

Results (perplexity + achieved ε) are saved to `results/dp_results.json` after each run.
Models are saved in fp16 to `checkpoints/<run_name>/`.
Completed runs are skipped automatically on re-execution.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/UofT/CSC2412/dp-finetuned-llms'
print('Drive mounted. Project dir:', PROJECT_DIR)

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets opacus accelerate

## 3. Imports & Config

In [ ]:
import os
import gc
import json
import math
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import (
    GPT2TokenizerFast,
    GPT2LMHeadModel,
    default_data_collator,
)
from datasets import load_from_disk
from opacus import PrivacyEngine

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ── Config ─────────────────────────────────────────────────────────────────────
EPSILONS      = [0.5, 1.0, 2.0, 4.0, 8.0, float('inf')]
CANARY_FREQS  = [1, 5, 10, 50]
DELTA         = 1e-4      # DP delta; 1/N for N ~ 10 000
MAX_GRAD_NORM = 1.0       # gradient clipping bound C
BATCH_SIZE    = 8         # small batch: Opacus allocs batch x vocab x emb_dim per step
LR            = 5e-5
EPOCHS        = 3
SEED          = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
RESULTS_PATH = f'{PROJECT_DIR}/results/dp_results.json'

## 4. Helper Functions

In [ ]:
def make_loaders(train_ds, val_ds):
    train_ds.set_format('torch')
    val_ds.set_format('torch')
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=default_data_collator, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=default_data_collator,
    )
    return train_loader, val_loader


def load_fresh_model(tokenizer):
    """Load GPT-2 small, fix for Opacus compatibility, untie weights."""
    model = GPT2LMHeadModel.from_pretrained('gpt2')
    model.resize_token_embeddings(len(tokenizer))
    # Untie lm_head from embedding (Opacus cannot handle shared parameters)
    model.lm_head.weight = torch.nn.Parameter(model.lm_head.weight.detach().clone())
    return model


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100  # ignore padding in loss

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()

        total_loss += outputs.loss.item()
        n += 1
    return total_loss / n


def evaluate_perplexity(model, val_loader):
    # Unwrap Opacus wrapper for inference
    base = model._module if hasattr(model, '_module') else model
    base.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100
            outputs = base(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            n += 1
    return math.exp(total_loss / n)


def save_model_fp16(model, tokenizer, path):
    os.makedirs(path, exist_ok=True)
    base = model._module if hasattr(model, '_module') else model
    base.half().save_pretrained(path)  # save in fp16 to save Drive space
    tokenizer.save_pretrained(path)


def is_done(results, epsilon, freq):
    eps_key = None if epsilon == float('inf') else epsilon
    return any(r['epsilon_target'] == eps_key and r['freq'] == freq for r in results)


print('Helper functions defined.')

## 5. Training Grid

Iterates over all 24 (ε, frequency) conditions.
Completed runs are skipped — safe to re-run after an interruption.

In [ ]:
# Load existing results (resume support)
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f'Loaded {len(results)} existing results from {RESULTS_PATH}')
else:
    results = []
    print('Starting fresh — no existing results found.')

total_runs  = len(EPSILONS) * len(CANARY_FREQS)
done_runs   = sum(1 for eps in EPSILONS for freq in CANARY_FREQS if is_done(results, eps, freq))
print(f'Progress: {done_runs}/{total_runs} runs completed.')

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained(f'{PROJECT_DIR}/data/tokenizer')
val_ds    = load_from_disk(f'{PROJECT_DIR}/data/val_ds')

for epsilon in EPSILONS:
    for freq in CANARY_FREQS:

        if is_done(results, epsilon, freq):
            eps_str = 'inf' if epsilon == float('inf') else str(epsilon)
            print(f'[skip] ε={eps_str}  freq={freq}x  (already done)')
            continue

        eps_str  = 'inf' if epsilon == float('inf') else str(epsilon)
        run_name = f'eps{eps_str}_freq{freq}'
        print(f'\n{"="*55}')
        print(f'  Run: {run_name}  |  ε={eps_str}  freq={freq}x')
        print(f'{"="*55}')

        # Load dataset for this frequency
        train_ds = load_from_disk(f'{PROJECT_DIR}/data/train_ds_canary_freq{freq}')
        train_loader, val_loader = make_loaders(train_ds, val_ds)

        # Fresh model + optimizer
        model     = load_fresh_model(tokenizer).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

        if epsilon == float('inf'):
            # ── No DP ────────────────────────────────────────────────────────
            for epoch in range(EPOCHS):
                avg_loss = train_one_epoch(model, train_loader, optimizer)
                print(f'  epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}')
            achieved_eps = None

        else:
            # ── DP-SGD via Opacus ────────────────────────────────────────────
            model.train()  # Opacus requires train mode before wrapping
            privacy_engine = PrivacyEngine()
            model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
                module=model,
                optimizer=optimizer,
                data_loader=train_loader,
                epochs=EPOCHS,
                target_epsilon=epsilon,
                target_delta=DELTA,
                max_grad_norm=MAX_GRAD_NORM,
                grad_sample_mode="ew",
            )
            sigma = optimizer.noise_multiplier
            print(f'  noise_multiplier σ={sigma:.4f}')

            for epoch in range(EPOCHS):
                avg_loss = train_one_epoch(model, train_loader, optimizer)
                eps_so_far = privacy_engine.get_epsilon(DELTA)
                print(f'  epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}  ε_so_far={eps_so_far:.3f}')

            achieved_eps = privacy_engine.get_epsilon(DELTA)
            print(f'  Final ε achieved: {achieved_eps:.4f} (target: {epsilon})')

        # Evaluate perplexity
        ppl = evaluate_perplexity(model, val_loader)
        print(f'  Perplexity: {ppl:.2f}')

        # Save model in fp16
        model_path = f'{PROJECT_DIR}/checkpoints/{run_name}'
        save_model_fp16(model, tokenizer, model_path)
        print(f'  Model saved to {model_path}')

        # Record result
        results.append({
            'run':              run_name,
            'epsilon_target':   None if epsilon == float('inf') else epsilon,
            'epsilon_achieved': achieved_eps,
            'freq':             freq,
            'perplexity':       ppl,
            'model_path':       model_path,
        })
        with open(RESULTS_PATH, 'w') as f:
            json.dump(results, f, indent=2)

        # Free GPU memory
        del model, optimizer, train_ds, train_loader, val_loader
        if 'privacy_engine' in dir():
            del privacy_engine
        torch.cuda.empty_cache()
        gc.collect()

print('\nAll runs complete.')

## 6. Results Summary

In [ ]:
with open(RESULTS_PATH) as f:
    results = json.load(f)

print(f'=== DP Training Results ({len(results)} runs) ===')
print(f'{"Run":<22}  {"ε target":>9}  {"ε achieved":>11}  {"freq":>5}x  {"Perplexity":>11}')
print('-' * 68)
for r in sorted(results, key=lambda x: (x['epsilon_target'] or float('inf'), x['freq'])):
    eps_t = f"{r['epsilon_target']:.1f}" if r['epsilon_target'] is not None else 'inf'
    eps_a = f"{r['epsilon_achieved']:.4f}" if r['epsilon_achieved'] is not None else 'inf'
    print(f"{r['run']:<22}  {eps_t:>9}  {eps_a:>11}  {r['freq']:>5}x  {r['perplexity']:>11.2f}")

---
**Next:** Run `5_evaluate.ipynb` to compute canary exposure scores and generate plots.